# Subagent - 영화 추천 시스템
* main agent
1. 사용자의 입력을 받아 처리
2. 사용자의 영화 장르 저장
3. 저장된 사용자의 영화 장르 불러오기
4. 사용자에게 영화 장르 기반 추천 -> Subagent

In [ ]:
# uv add -U langchain langgraph langchain-openai langchain-community duckduckgo-search

In [2]:
from dataclasses import dataclass
from typing import Any

from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from langchain_community.tools import DuckDuckGoSearchResults

In [5]:
# -----------------------------------
# 1. 사용자 context
# -----------------------------------
@dataclass
class UserContext:
    user_id: str


# -----------------------------------
# 2. 장기 메모리 저장소
# -----------------------------------
store = InMemoryStore()


# -----------------------------------
# 3. DuckDuckGo 검색 도구 준비
# -----------------------------------
# output_format="list" 로 받으면 검색 결과를 구조화된 형태로 다루기 쉽습니다.
ddg_tool = DuckDuckGoSearchResults(
    output_format="list",
    max_results=8,
)


# -----------------------------------
# 4. 좋아하는 영화 장르 저장 tool
# -----------------------------------
@tool
def save_favorite_genre(genre: str, runtime: ToolRuntime[UserContext]) -> str:
    """
    사용자가 좋아하는 영화 장르를 장기 메모리에 저장합니다.
    예: genre='sci-fi'
    """
    assert runtime.store is not None

    namespace = ("users", runtime.context.user_id, "movie_preferences")

    runtime.store.put(
        namespace,
        "favorite_genre",
        {"genre": genre}
    )

    return f"좋아하는 영화 장르를 '{genre}'로 저장했습니다."


# -----------------------------------
# 5. 저장된 장르 조회 tool
# -----------------------------------
@tool
def load_favorite_genre(runtime: ToolRuntime[UserContext]) -> str:
    """
    저장된 영화 장르를 조회합니다.
    """
    assert runtime.store is not None

    namespace = ("users", runtime.context.user_id, "movie_preferences")
    memory = runtime.store.get(namespace, "favorite_genre")

    if memory is None:
        return "아직 저장된 좋아하는 영화 장르가 없습니다."

    return f"저장된 좋아하는 영화 장르는 '{memory.value['genre']}' 입니다."


# -----------------------------------
# 6. 최근 영화 검색 + 추천 전용 서브에이전트
# -----------------------------------
movie_recommender_agent = create_agent(
    model="openai:gpt-5.4-nano",
    tools=[],
    system_prompt=(
        "당신은 영화 추천 전문가입니다. "
        "입력으로 사용자의 선호 장르와 최신 웹 검색 결과가 주어집니다. "
        "검색 결과에 나온 최근 영화들 중에서 그 장르에 맞는 작품 3~5편을 골라 추천하세요. "
        "반드시 아래 원칙을 따르세요:\n"
        "1. 검색 결과에 근거해 추천하세요.\n"
        "2. 검색 결과에 없는 영화를 지어내지 마세요.\n"
        "3. 각 영화마다 추천 이유를 1문장씩 쓰세요.\n"
        "4. 마지막에 '왜 이 영화를 골랐는지'를 짧게 요약하세요.\n"
        "5. 답변은 한국어로 하세요."
    ),
)


# -----------------------------------
# 7. 검색 기반 추천 tool
# -----------------------------------
@tool
def recommend_recent_movies(runtime: ToolRuntime[UserContext]) -> str:
    """
    저장된 좋아하는 장르를 바탕으로 DuckDuckGo에서 최근 영화를 검색한 뒤 추천합니다.
    """
    assert runtime.store is not None

    namespace = ("users", runtime.context.user_id, "movie_preferences")
    memory = runtime.store.get(namespace, "favorite_genre")

    if memory is None:
        return "좋아하는 영화 장르가 저장되어 있지 않습니다. 먼저 좋아하는 장르를 알려주세요."

    genre = memory.value["genre"]

    # 최근작 중심으로 검색
    search_query = (
        f"넷플릭스영화중 {genre}장르의 영화를 찾아줘. 영화 이름하고 장르로 데이터를 가져와줘. "
    )

    try:
        search_results = ddg_tool.invoke(search_query)
    except Exception as e:
        return f"영화 검색 중 오류가 발생했습니다: {e}"

    if not search_results:
        return f"'{genre}' 장르에 대한 최근 영화 검색 결과를 찾지 못했습니다."

    
    print('덕덕고 검색 결과:',search_results)
    result = movie_recommender_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        f"사용자가 좋아하는 영화 장르는 '{genre}' 입니다.\n\n"
                        f"아래는 DuckDuckGo로 찾은 최근 영화 관련 검색 결과입니다.\n\n"
                        f"{search_results}\n\n"
                        f"이 자료를 바탕으로 추천해주세요."
                    ),
                }
            ]
        }
    )

    return result["messages"][-1].content


# -----------------------------------
# 8. 메인 에이전트
# -----------------------------------
agent = create_agent(
    model="gpt-5.4-mini",
    tools=[save_favorite_genre, load_favorite_genre, recommend_recent_movies],
    store=store,
    context_schema=UserContext,
    system_prompt=(
        "당신은 사용자의 영화 취향을 기억하고 추천해주는 도우미입니다.\n"
        "- 사용자가 좋아하는 영화 장르를 말하면 save_favorite_genre를 사용하세요.\n"
        "- 사용자가 저장된 장르를 물어보면 load_favorite_genre를 사용하세요.\n"
        "- 사용자가 영화 추천을 요청하면 recommend_recent_movies를 사용하세요.\n"
        "- 답변은 한국어로 하세요."
    ),
)


# -----------------------------------
# 9. 실행 예시
# -----------------------------------
user_context = UserContext(user_id="user_001")

# 1) 장르 저장
response1 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "나는 영화 액션 장르를 좋아해. 기억해줘."}
        ]
    },
    context=user_context,
)
print("=== 응답 1 ===")
print(response1["messages"][-1].content)

# 2) 장르 확인
response2 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "내가 좋아하는 영화 장르가 뭐였지?"}
        ]
    },
    context=user_context,
)
print("\n=== 응답 2 ===")
print(response2["messages"][-1].content)

# 3) 최근 영화 추천
response3 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "내 취향에 맞는 최근 영화 추천해줘."}
        ]
    },
    context=user_context,
)
print("\n=== 응답 3 ===")
print(response3["messages"][-1].content)

=== 응답 1 ===
좋아요, 영화 **액션** 장르를 기억해둘게요.  
나중에 추천이 필요하면 말씀해 주세요!

=== 응답 2 ===
저장된 좋아하는 영화 장르는 **액션**입니다.
덕덕고 검색 결과: [{'snippet': '개인적으로도 액션 영화를 좋아해서 그동안 본 작품중에서 킬링타임으로 좋은 영화들 위주로 준비해보았으며 평점과 줄거리 한줄평을 간단히 정리해 두었기 때문에 본인에게 맞는 작품들을 쉽게 찾을 수 있으리라 생각됩니다. 그럼 아래에서 바로 ...', 'title': '넷플릭스 액션 영화 추천 Top40 - 짜릿한 한방', 'link': 'https://dreaminfo.tistory.com/1130'}, {'snippet': '안녕하세요, 힐스터K 입니다. 액션 영화는 단순한 폭력의 나열이 아닌, 인물의 내면과 세계관을 압축적으로 표현하는 강렬한 언어다. 아래 넷플릭스에서 감상할 수 있는 다섯 편의 액션 영화를 소개해 볼까 합니다.', 'title': '피도 눈물도 없는 넷플릭스 액션 영화 추천 5편 - 네이버 블로그', 'link': 'https://m.blog.naver.com/tmzmfk010/223943983442'}, {'snippet': '필터를 사용하여 Netflix의 새로운 액션영화를 확인하세요. 장르별, 개봉연도, 평점, 연령등급 등으로도 필터링하여 지금 바로 시청할 수 있는 Netflix의 최고 신작을 쉽게 찾을 수 있습니다. Netflix의 최신 영화를 찾으세요? JustWatch는 Netflix에 추가된 모든 새로운 영화의 전체 목록을 제공합니다!', 'title': 'Netflix의 최신 영화 - 최근에 추가된 모든 영화 - JustWatch', 'link': 'https://www.justwatch.com/kr/동영상서비스/netflix/최신/영화산업'}, {'snippet': '넷플릭스에서 즐길 수 있는 다양한 액션 영화 중 어떤 작품을 선택해야 할지 고민이신가요? 이 글에서는 엄선된 넷플릭스 액션 영화 추천 인